# Khmer-LLaDA-Small — train on Kaggle (T4)

From-scratch **masked diffusion** language model for Khmer, trained on a single Kaggle T4 (16 GB).
This notebook drives the [`Pich09/Khmer-LLaDA-Small`](https://github.com/Pich09/Khmer-LLaDA-Small) repo end to end.

**Before you run — Notebook settings (right sidebar):**
1. **Accelerator → GPU T4 x2** (only one GPU is used) or **P100**.
2. **Internet → On** (needed to clone the repo and pull the tokenizer + corpus).
3. *(optional, for cross-session resume)* **Add-ons → Secrets → add `HF_TOKEN`**.

**Pipeline:** clone → deps → tokenizer + corpus → pre-tokenize into packed shards → unit tests + overfit gate (Milestone-1) → train (token-scheduled, resumable) → sample + loss curves.

**`SMOKE = True` (cell 1) is the default** — a tiny end-to-end check (`SMOKE_TOKENS`, default 64 000) on a truncated corpus, a few minutes total. Flip it to `False` for a real token-scheduled run.

Kaggle sessions cap at ~12 h. With a write-scoped `HF_TOKEN`, every checkpoint is pushed to **`Panhapich/Khmer-LLaDA-Small/checkpoints/`** and pulled back automatically next session (section 16). Without a token, checkpoints stay on `/kaggle/working` — use *Save Version*.

## 1 · Environment

In [ ]:
import os, sys, subprocess, platform, torch

print('python :', platform.python_version())
print('torch  :', torch.__version__)
print('cuda   :', torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'gpu    : {p.name}  {p.total_memory/1e9:.1f} GB  sm_{p.major}{p.minor}')
    if p.total_memory < 14e9:
        print('NOTE: <16 GB card — drop PHYSICAL_BATCH to 4 (or 2) in the training config cell.')
else:
    print('WARNING: no GPU — turn on the accelerator, or expect training to be unusably slow.')

WORK     = '/kaggle/working'
REPO_DIR = os.path.join(WORK, 'Khmer-LLaDA-Small')
CKPT_DIR = os.path.join(WORK, 'checkpoints')
os.makedirs(CKPT_DIR, exist_ok=True)

# -------- run mode --------
SMOKE        = True      # True: tiny end-to-end pipeline check.  False: real token-scheduled run.
SMOKE_TOKENS = 64_000    # (SMOKE only) total tokens to train on for the check
print('SMOKE:', SMOKE, '| SMOKE_TOKENS:', SMOKE_TOKENS if SMOKE else '-')

## 2 · Clone the repo

In [ ]:
REPO_URL = 'https://github.com/Pich09/Khmer-LLaDA-Small.git'

if not os.path.isdir(os.path.join(REPO_DIR, '.git')):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=False)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print('cwd:', os.getcwd())
print(subprocess.run(['git', '-C', REPO_DIR, 'log', '--oneline', '-1'],
                     capture_output=True, text=True).stdout.strip())

## 3 · Dependencies

`torch` / `numpy` / `pyyaml` / `tqdm` / `matplotlib` ship with the Kaggle image. Only the tokenizer stack is added, pinned to the repo's `requirements.txt`. `khmer-nltk` / `wordninja` are only needed for encoding **raw** Khmer prompts in the sampling cell — training uses the pre-segmented corpus and does not need them, so their install is best-effort.

In [ ]:
# critical for training (SentencePiece over the pre-segmented corpus)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sentencepiece==0.2.0'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'huggingface_hub>=0.24,<1.0'], check=True)

# best-effort: only used by the raw-prompt path in the sampling cell
try:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'khmer-nltk==1.6', 'wordninja==2.0.0'], check=True)
    RAW_TOKENIZER_OK = True
except subprocess.CalledProcessError as e:
    print('khmer-nltk/wordninja install failed — sampling will fall back to unconditional only:', e)
    RAW_TOKENIZER_OK = False

import importlib
importlib.import_module('sentencepiece')
import yaml, numpy as np, glob, json, math
print('deps OK  (raw-prompt tokenizer available:', RAW_TOKENIZER_OK, ')')

## 4 · Hugging Face token + checkpoint repo

The tokenizer and corpus are public, so training works with **no token**. A **write-scoped** `HF_TOKEN` (Kaggle → Add-ons → Secrets) enables the main persistence path: every checkpoint is pushed to `checkpoints/` in the HF model repo below, and a fresh session pulls the latest back automatically. Without a token, checkpoints live only on `/kaggle/working` (use *Save Version* to keep them).

In [ ]:
HF_TOKEN = ''
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN') or ''
except Exception:
    pass
HF_TOKEN = HF_TOKEN or os.environ.get('HF_TOKEN', '')

HUB_CKPT_REPO = 'Panhapich/Khmer-LLaDA-Small'   # pretrain checkpoints go here (needs write access)

os.environ.pop('HUB_CKPT_REPO', None)
if HF_TOKEN:
    from huggingface_hub import login, create_repo
    login(token=HF_TOKEN, add_to_git_credential=False)
    os.environ['HF_TOKEN'] = HF_TOKEN
    if SMOKE:
        print('HF: logged in — but SMOKE run: Hub checkpoint push/pull is DISABLED (would clobber the real repo).')
    elif HUB_CKPT_REPO:
        create_repo(HUB_CKPT_REPO, repo_type='model', exist_ok=True, token=HF_TOKEN)
        os.environ['HUB_CKPT_REPO'] = HUB_CKPT_REPO   # train.py reads this and pushes every save
        print(f'HF: logged in — checkpoints -> {HUB_CKPT_REPO}/checkpoints/')
    else:
        print('HF: logged in (HUB_CKPT_REPO not set — no Hub checkpointing)')
else:
    HUB_CKPT_REPO = ''
    print('HF: no token — checkpoints stay on /kaggle/working only (use Save Version to persist).')

HUB_ACTIVE = bool(HF_TOKEN and HUB_CKPT_REPO and not SMOKE)   # Hub checkpointing on?

## 5 · Download tokenizer + corpus

Tokenizer files land in `<repo>/tokenizer/`, the corpus in `<repo>/data/raw/` — exactly where the repo's scripts expect them. Only `all_text_segmented.txt` is pulled (already khmer-nltk word-segmented, so a bare SentencePiece pass is correct **and** fast); the raw `all_text.txt` is skipped. The cell then asserts the SentencePiece model is **vocab 8000 with specials `<PAD>=0 <UNK>=1 <BOS>=2 <EOS>=3 <MASK>=4`** — the values every config and `khmer_llada/constants.py` hardcode; a mismatch here would corrupt training silently.

In [ ]:
from huggingface_hub import hf_hub_download

TOK_DIR = os.path.join(REPO_DIR, 'tokenizer')
RAW_DIR = os.path.join(REPO_DIR, 'data', 'raw')
os.makedirs(TOK_DIR, exist_ok=True)
os.makedirs(RAW_DIR, exist_ok=True)

for f in ['khmer_sp.model', 'khmer_sp.vocab', 'khmer_segmentation.py',
          'gazetteer.json', 'latin_exceptions.json', 'tokenizer_info.json', 'USAGE.md']:
    try:
        hf_hub_download('Panhapich/khmer-sp-8k', f, repo_type='model',
                        local_dir=TOK_DIR, token=HF_TOKEN or None)
    except Exception as e:
        print(f'  (skip {f}: {e})')
assert os.path.exists(os.path.join(TOK_DIR, 'khmer_sp.model')), 'khmer_sp.model missing — cannot continue'
print('tokenizer:', sorted(os.listdir(TOK_DIR)))

# verify the tokenizer is exactly what every config / constants.py assumes: vocab 8000, specials 0..4
import sentencepiece as spm
_sp = spm.SentencePieceProcessor(model_file=os.path.join(TOK_DIR, 'khmer_sp.model'))
TOK_VOCAB   = _sp.get_piece_size()
TOK_SPECIAL = [_sp.piece_to_id(t) for t in ('<PAD>', '<UNK>', '<BOS>', '<EOS>', '<MASK>')]
print(f'tokenizer vocab: {TOK_VOCAB}   PAD/UNK/BOS/EOS/MASK ids: {TOK_SPECIAL}')
assert TOK_VOCAB == 8000, f'khmer-sp-8k should be vocab 8000, got {TOK_VOCAB}'
assert TOK_SPECIAL == [0, 1, 2, 3, 4], 'special-token ids differ from the fixed 0..4 the repo hardcodes'

CORPUS_FILE = os.path.join(RAW_DIR, 'all_text_segmented.txt')
if not os.path.exists(CORPUS_FILE):
    hf_hub_download('Panhapich/khmer-text-corpus', 'all_text_segmented.txt',
                    repo_type='dataset', local_dir=RAW_DIR, token=HF_TOKEN or None)

n_lines = sum(1 for _ in open(CORPUS_FILE, encoding='utf-8'))
print(f'corpus: {CORPUS_FILE}  {os.path.getsize(CORPUS_FILE)/1e6:.0f} MB  {n_lines:,} lines')

# SMOKE: pre-tokenize only a slice so the whole notebook runs in a few minutes
CORPUS = CORPUS_FILE
if SMOKE:
    CORPUS = os.path.join(RAW_DIR, 'smoke_corpus.txt')
    if not os.path.exists(CORPUS):
        with open(CORPUS_FILE, encoding='utf-8') as fi, open(CORPUS, 'w', encoding='utf-8') as fo:
            for i, ln in enumerate(fi):
                if i >= 20_000:
                    break
                fo.write(ln)
    print('SMOKE: using first 20k lines ->', CORPUS)

## 6 · Pre-tokenize → packed `uint16` shards

Concatenates ids + `<EOS>`, chunks into fixed `SEQ_LEN` windows (no padding), writes memory-mapped shards and a held-out `val` split (last 5 000 lines). SMOKE writes to `data/shards_smoke/`, a real run to `data/shards/` — so flipping `SMOKE` never mixes the two. Re-runs skip if shards exist.

In [ ]:
SEQ_LEN   = 512
SHARD_DIR = os.path.join(REPO_DIR, 'data', 'shards_smoke' if SMOKE else 'shards')

have_shards = os.path.isdir(SHARD_DIR) and any(
    n.startswith('train_') and n.endswith('.npy') for n in os.listdir(SHARD_DIR))
if not have_shards:
    subprocess.run([sys.executable, 'scripts/pretokenize.py',
                    '--in', CORPUS, '--seq-len', str(SEQ_LEN),
                    '--out-dir', SHARD_DIR], check=True)
else:
    print('shards already present — skipping pretokenize')

def _tok_count(split):
    return sum(np.load(p, mmap_mode='r').shape[0]
               for p in glob.glob(os.path.join(SHARD_DIR, f'{split}_*.npy'))) * SEQ_LEN
train_tok, val_tok = _tok_count('train'), _tok_count('val')
print(f'packed train tokens: {train_tok:,}   val tokens: {val_tok:,}')
assert train_tok > 0 and val_tok > 0, 'pre-tokenization produced no data'

## 7 · Pick the model size

Auto-picked from the packed token budget (PLAN §1b: ~50–100 tokens/param). **For the pretrain → ASR plan, set `FORCE_MODEL_CFG = 'configs/small_b.json'`** — hidden 768 matches `openai/whisper-small`'s encoder dim, so the downstream ASR repo needs no projection layer between audio and the decoder's cross-attention.

In [ ]:
from khmer_llada import Config

FORCE_MODEL_CFG = ''   # '' = auto.  Set 'configs/small_b.json' for the ASR plan (hidden 768 = whisper-small).

# tiny.json (ctx 128) is the overfit-gate model only — never auto-picked for a real run.
if FORCE_MODEL_CFG:
    MODEL_CFG, why = FORCE_MODEL_CFG, f'forced -> {FORCE_MODEL_CFG}'
elif train_tok < 1_000_000_000:
    MODEL_CFG, why = 'configs/small_a.json', f'{train_tok/1e9:.2f} B tokens -> Small-A (~64 M); plan on 3-4 epochs'
elif train_tok < 3_000_000_000:
    MODEL_CFG, why = 'configs/small_b.json', f'{train_tok/1e9:.2f} B tokens -> Small-B (~91 M)'
else:
    MODEL_CFG, why = 'configs/small_b.json', f'{train_tok/1e9:.2f} B tokens -> Small-B (or add a Small-C config)'

_c  = Config.from_json(MODEL_CFG)
_pc = _c.param_count()
assert _c.max_position_embeddings >= SEQ_LEN, (
    f'{MODEL_CFG} ctx {_c.max_position_embeddings} < SEQ_LEN {SEQ_LEN}')
assert _c.vocab_size == TOK_VOCAB, f'config vocab_size {_c.vocab_size} != tokenizer {TOK_VOCAB}'
if train_tok < 150_000_000 and not SMOKE:
    print('WARNING: very small corpus — this will be a methods demo, not a usable LM. '
          'Add more Khmer sources (PLAN §4a) before a real run.')
print(why)
print(f'-> {MODEL_CFG}   ~{_pc["total_millions"]} M params   '
      f'tokens/param at 1 epoch: {train_tok/_pc["total"]:.1f}')

## 8 · Unit tests + overfit gate (Milestone-1)

The three GPU-free tests assert bidirectional attention, the `1/t` diffusion loss, and bit-exact checkpoint resume. The **overfit gate** then trains Tiny on ~800 sequences: real run needs masked loss < 0.1 nats/token in 4000 steps; SMOKE only checks < 3.0 in 800 steps (proves the loop runs). **If the gate fails, stop** — the bug is in the loss / mask / attention path and no amount of GPU time will fix it.

In [ ]:
subprocess.run([sys.executable, '-m', 'pytest', '-q',
                'tests/test_bidirectional.py', 'tests/test_diffusion.py',
                'tests/test_resume.py'], check=True)

In [ ]:
subprocess.run([sys.executable, 'scripts/make_overfit_set.py',
                '--in', CORPUS, '--out', 'data/overfit.npy', '--n', '800'], check=True)

# SMOKE: short/loose check that the training primitives run. The real gate is 4000 steps -> 0.1;
# run that (SMOKE=False, or bump the args) before trusting a full training run.
_steps, _target = ('800', '3.0') if SMOKE else ('4000', '0.1')
subprocess.run([sys.executable, 'training/overfit.py',
                '--data', 'data/overfit.npy', '--model-config', 'configs/tiny.json',
                '--steps', _steps, '--target', _target], check=True)   # non-zero exit == gate failed

## 9 · Training configuration

Writes a tuned copy of `configs/train_t4.yaml` to `/kaggle/working`, with `ckpt_dir` on `/kaggle/working/checkpoints`.

| | **SMOKE** (`SMOKE=True`) | **real** (`SMOKE=False`) |
|---|---|---|
| total tokens | `SMOKE_TOKENS` (64 000) | `TOTAL_TOKENS` 300 M — raise to 2 B+ |
| batch × accum | 1 × 1 | 8 × 32 |
| warmup / save / val | 10 / 20 / 20 steps | 100 / 500 / 500 steps |

`warmup`/`save`/`val` auto-shrink further if `total_steps` is tiny, so they always fire. LR: linear `0 -> lr` over `warmup_steps`, then cosine to `min_lr` (10 % of peak) over `total_steps`.

The LR is recomputed from `step` every iteration (no scheduler state is stored in the checkpoint), so resume lands exactly on the curve — **but `TOTAL_TOKENS`/`SMOKE_TOKENS`, `PHYSICAL_BATCH` and `GRAD_ACCUM` must stay identical across resume sessions**, or `total_steps` shifts and both the cosine schedule and the data-loader position desync. If the profile cell OOMs, lower `PHYSICAL_BATCH` and **restart** rather than resume.

In [ ]:
if SMOKE:
    TOTAL_TOKENS, PHYSICAL_BATCH, GRAD_ACCUM = SMOKE_TOKENS, 1, 1
    WARMUP, SAVE_EVERY, VBATCH, VSAMP = 10, 20, 4, 8
    RUN_NAME = 'kaggle_smoke'
else:
    TOTAL_TOKENS, PHYSICAL_BATCH, GRAD_ACCUM = 300_000_000, 8, 32   # raise TOTAL_TOKENS to 2e9+ for a full run
    WARMUP, SAVE_EVERY, VBATCH, VSAMP = 100, 500, 20, 32
    RUN_NAME = 'kaggle_run1'

tokens_per_step = PHYSICAL_BATCH * GRAD_ACCUM * SEQ_LEN
total_steps     = max(1, TOTAL_TOKENS // tokens_per_step)

train_cfg = dict(
    seq_len=SEQ_LEN,
    lr=3.0e-4, min_lr=3.0e-5,
    warmup_steps=min(WARMUP, max(1, total_steps // 5)),
    weight_decay=0.1, beta1=0.9, beta2=0.95, grad_clip_norm=1.0,
    physical_batch=PHYSICAL_BATCH, grad_accum=GRAD_ACCUM,
    total_tokens=int(TOTAL_TOKENS),
    precision='fp16',
    gradient_checkpointing=False, adam_8bit=False,
    data_glob=os.path.join(SHARD_DIR, 'train_*.npy'),
    val_glob=os.path.join(SHARD_DIR, 'val_*.npy'),
    ckpt_dir=CKPT_DIR,
    save_every_steps=min(SAVE_EVERY, max(1, total_steps // 4)),
    val_every_steps=min(SAVE_EVERY, max(1, total_steps // 4)),
    log_every_steps=max(1, min(25, total_steps // 10)),
    val_batches=VBATCH, val_nll_samples=VSAMP,
    seed=42,
    wandb=False, wandb_project='khmer-llada-small', run_name=RUN_NAME,
)
TRAIN_CFG = os.path.join(WORK, 'train_kaggle.yaml')
yaml.safe_dump(train_cfg, open(TRAIN_CFG, 'w'), sort_keys=False)
print(open(TRAIN_CFG).read())
print(f'derived: total_steps ~{total_steps:,}   tokens/step {tokens_per_step:,}   '
      f'warmup {train_cfg["warmup_steps"]}   save/val every {train_cfg["save_every_steps"]} steps   '
      f'~{TOTAL_TOKENS/max(1,train_tok):.2f} epochs over the train split')

def _lr_at(s, w=train_cfg['warmup_steps'], T=total_steps,
           pk=train_cfg['lr'], fl=train_cfg['min_lr']):
    if s < w:
        return pk * s / max(1, w)
    p = min(1.0, (s - w) / max(1, T - w))
    return fl + 0.5 * (pk - fl) * (1 + math.cos(math.pi * p))
print('LR curve : ' + '   '.join(
    f'{lbl} @{s}: {_lr_at(s):.2e}' for lbl, s in
    [('start', 0), ('warmup-end', train_cfg['warmup_steps']),
     ('mid', total_steps // 2), ('end', total_steps)]))

## 10 · Patch `train.py` — `best.pt` + push each save to the Hub

The repo's loop only writes a rolling local `last.pt` (+ `final.pt` at the end). This idempotent patch adds:

- **`best.pt`** — written in the validation branch whenever `val_nll_bound` improves; `best_val` is recovered from `metrics.jsonl` on start so it survives restarts.
- **Hub push** — after every `last.pt` / `best.pt` / `final.pt` save, the file (plus `metrics.jsonl`) is uploaded to `checkpoints/` in `HUB_CKPT_REPO` (`Panhapich/Khmer-LLaDA-Small`), when `os.environ['HUB_CKPT_REPO']` is set (cell 4). No token → local-only, no error.

All checkpoints carry the full resume metadata the repo already saves: `model` / `optimizer` / `scaler` / data position / `step` / `tokens_seen` / `model_config` / `train_config` / RNG.

In [ ]:
tp  = os.path.join(REPO_DIR, 'training', 'train.py')
src = open(tp, encoding='utf-8').read()

if '_hub_push' in src:
    print('training/train.py already patched')
else:
    a1 = '    metrics_f = open(os.path.join(run_dir, "metrics.jsonl"), "a")\n'
    a2 = ('            if wandb:\n'
          '                wandb.log(rec, step=step)\n'
          '            model.train()\n')
    a3 = ('        if step % tc["save_every_steps"] == 0:\n'
          '            ckpt.save(os.path.join(tc["ckpt_dir"], "last.pt"), model=model, optimizer=opt,\n'
          '                      scaler=scaler, loader=loader, step=step, tokens_seen=tokens_seen,\n'
          '                      model_config=asdict(cfg), train_config=tc)\n')
    a4 = ('    ckpt.save(os.path.join(tc["ckpt_dir"], "final.pt"), model=model, optimizer=opt,\n'
          '              scaler=scaler, loader=loader, step=step, tokens_seen=tokens_seen,\n'
          '              model_config=asdict(cfg), train_config=tc)\n')
    assert all(a in src for a in (a1, a2, a3, a4)), 'train.py layout changed — update the patch anchors'

    src = src.replace(a1, a1 +
        '    best_val = float("inf")\n'
        '    _mp = os.path.join(run_dir, "metrics.jsonl")\n'
        '    if os.path.exists(_mp):\n'
        '        for _l in open(_mp):\n'
        '            try:\n'
        '                _v = json.loads(_l).get("val_nll_bound")\n'
        '                if _v is not None:\n'
        '                    best_val = min(best_val, _v)\n'
        '            except Exception:\n'
        '                pass\n'
        '    print(f"best_val on start: {best_val}", flush=True)\n'
        '\n'
        '    def _hub_push(*paths):\n'
        '        repo = os.environ.get("HUB_CKPT_REPO")\n'
        '        if not repo:\n'
        '            return\n'
        '        try:\n'
        '            from huggingface_hub import HfApi\n'
        '            api = HfApi()\n'
        '            for _p in paths:\n'
        '                if os.path.exists(_p):\n'
        '                    api.upload_file(path_or_fileobj=_p,\n'
        '                                   path_in_repo="checkpoints/" + os.path.basename(_p),\n'
        '                                   repo_id=repo, repo_type="model")\n'
        '            print(f"  hub: pushed {[os.path.basename(p) for p in paths]} -> {repo}", flush=True)\n'
        '        except Exception as _e:\n'
        '            print(f"  hub push failed: {_e}", flush=True)\n')

    src = src.replace(a2,
        '            if wandb:\n'
        '                wandb.log(rec, step=step)\n'
        '            if vb < best_val:\n'
        '                best_val = vb\n'
        '                ckpt.save(os.path.join(tc["ckpt_dir"], "best.pt"), model=model, optimizer=opt,\n'
        '                          scaler=scaler, loader=loader, step=step, tokens_seen=tokens_seen,\n'
        '                          model_config=asdict(cfg), train_config=tc)\n'
        '                print(f"  new best val_nll_bound {vb:.4f} -> best.pt", flush=True)\n'
        '                _hub_push(os.path.join(tc["ckpt_dir"], "best.pt"))\n'
        '            model.train()\n')

    src = src.replace(a3, a3 +
        '            _hub_push(os.path.join(tc["ckpt_dir"], "last.pt"), _mp)\n')
    src = src.replace(a4, a4 +
        '    _hub_push(os.path.join(tc["ckpt_dir"], "final.pt"), _mp)\n')

    open(tp, 'w', encoding='utf-8').write(src)
    print('patched training/train.py')

print(subprocess.run(['git', '-C', REPO_DIR, 'diff', '--stat'], capture_output=True, text=True).stdout)

## 11 · VRAM profile — one real step before committing to the run

In [ ]:
import torch
from khmer_llada import Config, KhmerLLaDA, forward_process, diffusion_loss

_dev = 'cuda' if torch.cuda.is_available() else 'cpu'
_cfg = Config.from_json(MODEL_CFG)
_m = KhmerLLaDA(_cfg).to(_dev)
_opt = torch.optim.AdamW(_m.parameters(), lr=1e-4)
_x = torch.randint(5, _cfg.vocab_size, (PHYSICAL_BATCH, SEQ_LEN), device=_dev)
if _dev == 'cuda':
    torch.cuda.reset_peak_memory_stats()
with torch.autocast(device_type=_dev, dtype=torch.float16, enabled=(_dev == 'cuda')):
    xt, msk, p = forward_process(_x, _cfg.mask_token_id)
    loss = diffusion_loss(_m(xt), _x, msk, p)
loss.backward(); _opt.step()
if _dev == 'cuda':
    print(f'peak VRAM for 1 step @ batch {PHYSICAL_BATCH} seq {SEQ_LEN}: '
          f'{torch.cuda.max_memory_allocated()/1e9:.2f} GB / {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print(f'params: {_m.num_parameters()/1e6:.1f} M total  |  init loss {loss.item():.2f}  (~ln V = {math.log(_cfg.vocab_size):.2f})')
del _m, _opt, _x, xt, loss
if _dev == 'cuda':
    torch.cuda.empty_cache()

## 12 · Resume check

Finds a checkpoint to continue from, in order: (1) `checkpoints/last.pt` already in this session, (2) `checkpoints/last.pt` in `HUB_CKPT_REPO` — the normal cross-session path, pulled with `best.pt` + `metrics.jsonl`, (3) the newest `last.pt` under `/kaggle/input/**` (a previous run's notebook output added as an input dataset).

In [ ]:
import shutil

RESUME     = None
local_last = os.path.join(CKPT_DIR, 'last.pt')
_exp_dir   = os.path.join(REPO_DIR, 'experiments', RUN_NAME)

if os.path.exists(local_last):
    RESUME = local_last
    print('resuming from this session:', local_last)
elif HUB_ACTIVE:
    from huggingface_hub import hf_hub_download
    for nm in ('last.pt', 'best.pt', 'metrics.jsonl'):
        try:
            p = hf_hub_download(HUB_CKPT_REPO, f'checkpoints/{nm}', repo_type='model', token=HF_TOKEN)
            if nm == 'metrics.jsonl':
                os.makedirs(_exp_dir, exist_ok=True)
                shutil.copy(p, os.path.join(_exp_dir, 'metrics.jsonl'))
            else:
                shutil.copy(p, os.path.join(CKPT_DIR, nm))
            print(f'pulled {nm} from {HUB_CKPT_REPO}')
        except Exception as e:
            print(f'  no {nm} on the hub yet ({type(e).__name__})')
    if os.path.exists(local_last):
        RESUME = local_last

if RESUME is None:
    found = sorted(glob.glob('/kaggle/input/**/last.pt', recursive=True), key=os.path.getmtime)
    if found:
        shutil.copy(found[-1], local_last)
        RESUME = local_last
        print('carried last.pt from', found[-1])
        b = os.path.join(os.path.dirname(found[-1]), 'best.pt')
        if os.path.exists(b):
            shutil.copy(b, os.path.join(CKPT_DIR, 'best.pt'))
        mets = sorted(glob.glob('/kaggle/input/**/metrics.jsonl', recursive=True), key=os.path.getmtime)
        if mets:
            os.makedirs(_exp_dir, exist_ok=True)
            shutil.copy(mets[-1], os.path.join(_exp_dir, 'metrics.jsonl'))

print('resume from:', RESUME or '(fresh start)')

## 13 · Train

Token-scheduled custom loop: fp16 + `GradScaler` + grad-clip + cosine LR. Overwrites `checkpoints/last.pt` every `save_every_steps`, writes `checkpoints/best.pt` on each `val_nll_bound` improvement, and appends `experiments/<RUN_NAME>/metrics.jsonl`. `final.pt` is written once at the end. If the session times out, just re-run the notebook — the resume cell above picks `last.pt` back up.

In [ ]:
cmd = [sys.executable, 'training/train.py',
       '--model-config', MODEL_CFG, '--train-config', TRAIN_CFG]
if RESUME:
    cmd += ['--resume', RESUME]
print(' '.join(cmd), flush=True)
subprocess.run(cmd, check=True)

## 14 · Loss curves

In [ ]:
import json, matplotlib.pyplot as plt

mfile = os.path.join(REPO_DIR, 'experiments', RUN_NAME, 'metrics.jsonl')
rows  = [json.loads(l) for l in open(mfile)] if os.path.exists(mfile) else []
tr = [(r['step'], r['train_loss'])    for r in rows if 'train_loss' in r]
vl = [(r['step'], r['val_nll_bound']) for r in rows if 'val_nll_bound' in r]
lr = [(r['step'], r['lr'])            for r in rows if 'lr' in r]

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
if tr: ax[0].plot(*zip(*tr)); ax[0].set(title='train_loss', xlabel='step')
if vl: ax[1].plot(*zip(*vl), color='tab:red'); ax[1].set(title='val_nll_bound', xlabel='step')
if lr: ax[2].plot(*zip(*lr), color='tab:green'); ax[2].set(title='lr', xlabel='step')
for a in ax: a.grid(alpha=.3)
plt.tight_layout()
plt.savefig(os.path.join(WORK, 'training_curves.png'), dpi=110)
plt.show()
print(f'{len(tr)} train points, {len(vl)} val points')

## 15 · Sample from the checkpoint

Loads `best.pt` if present, else `final.pt`, else `last.pt` (override with `SAMPLE_CKPT`). Uses a **patched** sampler: the repo's `khmer_llada/generation.py` collapses to all-`<MASK>` at `temperature=0` because it never removes `<MASK>` / `<PAD>` from the candidate logits before `argmax`, and has no final fill pass. The `generate_fixed` below adds both and asserts the context bound. (Track the upstream fix.)

In [ ]:
import torch, torch.nn.functional as F, json
from khmer_llada import Config, KhmerLLaDA
from khmer_llada.constants import MASK_ID, EOS_ID, BOS_ID, PAD_ID

device = 'cuda' if torch.cuda.is_available() else 'cpu'
SAMPLE_CKPT = ''   # set a path to override the best -> final -> last preference
ckpt_path = SAMPLE_CKPT or next(
    (os.path.join(CKPT_DIR, n) for n in ('best.pt', 'final.pt', 'last.pt')
     if os.path.exists(os.path.join(CKPT_DIR, n))), '')
assert ckpt_path, 'no checkpoint in ' + CKPT_DIR
state = torch.load(ckpt_path, map_location='cpu')
mcfg  = Config(**state['model_config']) if isinstance(state.get('model_config'), dict) else Config.from_json(MODEL_CFG)
model = KhmerLLaDA(mcfg).to(device)
model.load_state_dict(state['model'])
model.eval()
print('loaded', ckpt_path, '| step', state.get('step'), '| tokens_seen', state.get('tokens_seen'))


@torch.no_grad()
def generate_fixed(model, prompt_ids, gen_len=96, steps=96, block_length=None,
                   temperature=0.0, remasking='low_confidence', device='cuda'):
    model.eval()
    if prompt_ids is None:
        prompt_ids = torch.tensor([[BOS_ID]], dtype=torch.long, device=device)
    prompt_ids = prompt_ids.to(device)
    P = prompt_ids.shape[1]
    assert P + gen_len <= model.config.max_position_embeddings, \
        f'P+gen_len={P+gen_len} > max_position_embeddings={model.config.max_position_embeddings}'
    block_length = block_length or gen_len
    assert gen_len % block_length == 0
    n_blocks = gen_len // block_length
    assert steps % n_blocks == 0
    spb = steps // n_blocks

    x = torch.full((1, P + gen_len), MASK_ID, dtype=torch.long, device=device)
    x[:, :P] = prompt_ids

    def _counts(n, s):
        base, rem = n // s, n % s
        return [base + (1 if i < rem else 0) for i in range(s)]

    for b in range(n_blocks):
        lo, hi = P + b * block_length, P + (b + 1) * block_length
        counts = _counts(int((x[:, lo:hi] == MASK_ID).sum()), spb)
        for i in range(spb):
            is_mask = x == MASK_ID
            if not is_mask.any():
                break
            logits = model(x).float()
            logits[..., MASK_ID] = -float('inf')   # never commit <MASK>
            logits[..., PAD_ID]  = -float('inf')   # never commit <PAD>
            if temperature > 0:
                probs = F.softmax(logits / temperature, dim=-1)
                x0 = torch.multinomial(probs[0], 1).squeeze(-1).unsqueeze(0)
            else:
                x0 = logits.argmax(-1)
            if remasking == 'low_confidence':
                conf = F.softmax(logits, dim=-1).gather(-1, x0.unsqueeze(-1)).squeeze(-1)
            else:
                conf = torch.rand_like(x0, dtype=torch.float)
            conf = torch.where(is_mask, conf, torch.full_like(conf, -float('inf')))
            conf[:, :lo] = -float('inf')
            conf[:, hi:] = -float('inf')
            x0 = torch.where(is_mask, x0, x)
            k = counts[i]
            if k > 0:
                sel = torch.topk(conf[0], k=k).indices
                x[0, sel] = x0[0, sel]
        leftover = (x[0, lo:hi] == MASK_ID).nonzero(as_tuple=True)[0]
        if leftover.numel():
            logits = model(x).float()
            logits[..., MASK_ID] = -float('inf')
            logits[..., PAD_ID]  = -float('inf')
            x[0, lo + leftover] = logits.argmax(-1)[0, lo + leftover]
        eos = (x[0, P:] == EOS_ID).nonzero(as_tuple=True)[0]
        if eos.numel():
            x[0, P + int(eos[0]) + 1:] = EOS_ID
            break
    return x[:, P:]

In [ ]:
from khmer_llada.tokenizer import KhmerSPTokenizer

enc_tok = None
if RAW_TOKENIZER_OK:
    try:
        enc_tok = KhmerSPTokenizer(mode='wrapper')          # raw Khmer -> ids (khmer-nltk)
    except Exception as e:
        print('wrapper tokenizer unavailable:', e)
dec_tok = KhmerSPTokenizer(mode='sp_direct')                 # ids -> text

PROMPTS = ['', 'ប្រទេសកម្ពុជា', 'នៅថ្ងៃនេះ', 'ការសិក្សា']
rows = []
for prm in PROMPTS:
    pid = None
    if prm:
        if enc_tok is None:
            print(f'skip prompt {prm!r} (no raw tokenizer)'); continue
        pid = torch.tensor([[BOS_ID] + enc_tok.encode(prm)], dtype=torch.long, device=device)
    g = generate_fixed(model, pid, gen_len=96, steps=96, temperature=0.0, device=device)
    txt = dec_tok.decode(g[0].tolist())
    rows.append({'prompt': prm, 'generation': txt})
    print(f'[{prm!r}] -> {txt}\n')

json.dump(rows, open(os.path.join(WORK, 'samples.json'), 'w'), ensure_ascii=False, indent=2)
print('saved /kaggle/working/samples.json')

## 16 · Continue across sessions

A Kaggle session is ~12 h and one T4 does roughly **1–1.5 B tokens** in that time, so a real 2 B+ run spans a few sessions.

### Primary path — `HUB_CKPT_REPO` (with a write `HF_TOKEN`)
Nothing to do. `train.py` pushed `last.pt` / `best.pt` / `metrics.jsonl` to `Panhapich/Khmer-LLaDA-Small/checkpoints/` on every save (cell 10 patch); the resume cell pulls them at the start of the next session. Just **Run All** each session.

### Fallback — Kaggle notebook output (no token)
1. **Save Version** → *Save & Run All* — commits `/kaggle/working` (incl. `checkpoints/`).
2. Next session → **Add Input** → *Notebook Output* → this notebook's latest version → **Run All**. The resume cell finds `last.pt` under `/kaggle/input/**`.

### Manual push (safety button)
Run this any time to force-sync the local `checkpoints/` folder to the Hub — e.g. right before you stop a session early.

In [ ]:
if HUB_ACTIVE:
    from huggingface_hub import HfApi
    HfApi().upload_folder(folder_path=CKPT_DIR, path_in_repo='checkpoints',
                          repo_id=HUB_CKPT_REPO, repo_type='model', token=HF_TOKEN,
                          commit_message=f'manual sync @ {RUN_NAME}')
    print('synced', CKPT_DIR, '->', HUB_CKPT_REPO + '/checkpoints/')
else:
    print('no HF_TOKEN / HUB_CKPT_REPO — use Save Version to persist /kaggle/working instead.')

## 17 · Export for the ASR repo

This repo is **pretraining only**. This cell packs a portable `khmer-llada-small-pretrained/` for the separate ASR fine-tuning repo to consume:

| file | contents |
|---|---|
| `model.pt` | bare `KhmerLLaDA` state_dict — no optimizer / scaler / RNG |
| `config.json` | model config (vocab 8000, hidden, layers, heads, ctx) |
| `meta.json` | `step`, `tokens_seen`, git commit, full key list, tokenizer id |
| `tokenizer/` | `khmer_sp.model` + segmentation files (self-contained) |

**Contract for the ASR repo:** build the conditional model as an *extension* of `khmer_llada/modeling.py` — same block (`attn_norm → attn.wq/wk/wv/wo → mlp_norm → mlp.w1/w2/w3`), adding only per-block `cross_norm` + `cross_attn` (text queries over frozen `openai/whisper-small` features). Then:

```python
m = KhmerLLaDAASR(Config(**json.load(open('config.json'))))
missing, unexpected = m.load_state_dict(torch.load('model.pt'), strict=False)
assert not unexpected
assert all('cross' in k for k in missing)   # ONLY the new audio layers are fresh
```

Pretrain with **`small_b`** (hidden 768) so it matches whisper-small's encoder dim with no projection layer.

In [ ]:
import shutil, torch, json

EXPORT_DIR = os.path.join(WORK, 'khmer-llada-small-pretrained')
os.makedirs(EXPORT_DIR, exist_ok=True)

src = next((os.path.join(CKPT_DIR, n) for n in ('best.pt', 'final.pt', 'last.pt')
           if os.path.exists(os.path.join(CKPT_DIR, n))), '')
assert src, 'no checkpoint to export — run training first'
st = torch.load(src, map_location='cpu')

torch.save(st['model'], os.path.join(EXPORT_DIR, 'model.pt'))
json.dump(st['model_config'], open(os.path.join(EXPORT_DIR, 'config.json'), 'w'), indent=2)
json.dump({
    'source_checkpoint': os.path.basename(src),
    'step': st.get('step'), 'tokens_seen': st.get('tokens_seen'),
    'git_commit': subprocess.run(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD'],
                                 capture_output=True, text=True).stdout.strip(),
    'tokenizer': 'Panhapich/khmer-sp-8k (vocab 8000, specials 0..4)',
    'param_keys': sorted(st['model'].keys()),
}, open(os.path.join(EXPORT_DIR, 'meta.json'), 'w'), indent=2, ensure_ascii=False)
shutil.copytree(TOK_DIR, os.path.join(EXPORT_DIR, 'tokenizer'), dirs_exist_ok=True)

print('exported ->', EXPORT_DIR)
for f in sorted(os.listdir(EXPORT_DIR)):
    print('  ', f)
print(f"{len(st['model'])} weight tensors  |  step {st.get('step')}  tokens_seen {st.get('tokens_seen'):,}")

# push the portable export to HUB_CKPT_REPO under pretrained/ (alongside checkpoints/)
EXPORT_HUB_REPO = HUB_CKPT_REPO   # or set another repo id; '' to skip
if HF_TOKEN and EXPORT_HUB_REPO:
    from huggingface_hub import HfApi, create_repo
    create_repo(EXPORT_HUB_REPO, repo_type='model', exist_ok=True, token=HF_TOKEN)
    HfApi().upload_folder(folder_path=EXPORT_DIR, path_in_repo='pretrained',
                          repo_id=EXPORT_HUB_REPO, repo_type='model', token=HF_TOKEN)
    print('pushed ->', EXPORT_HUB_REPO)

## Notes

- **Overfit gate is a hard stop.** If cell 8 raises, the model/loss is wrong — fix before spending GPU hours.
- **`gradient_checkpointing` in the config is currently a no-op** in the repo. Leave it `false`; if you need it for Small-B at seq 1024, it must be wired into `modeling.py` first.
- **`val_nll_bound`** is a Monte-Carlo upper bound (fixed seed), not a true perplexity — comparable across checkpoints and against a future AR baseline, not against standard LM perplexity.
- **Data pipeline is minimal**: the corpus is used as-is with only a last-5 000-lines val holdout; there is no dedup between train and val. Fine for a first run, tighten (PLAN §4b) before publishing numbers.
- **Sample text may carry segmentation spaces.** Decoding goes through `sp_direct`, which returns khmer-nltk-spaced pieces, not natural unspaced Khmer (PLAN §6c) — a de-segmentation step is still a repo TODO. The raw-prompt (`wrapper`) path also depends on `KhmerTokenizer`'s constructor signature, which the repo hasn't pinned; the sampling cell falls back to unconditional generation if it doesn't load.
- **Checkpoints**: `last.pt` every `save_every_steps`, `best.pt` on each `val_nll_bound` improvement, `final.pt` at the end — local in `/kaggle/working/checkpoints/` **and** pushed to `Panhapich/Khmer-LLaDA-Small/checkpoints/` (with a write `HF_TOKEN`). Each holds model + optimizer + scaler + data position + `step` + `tokens_seen` + `model_config` + `train_config` + RNG — enough to resume bit-exactly or rebuild the model standalone.
- **Hub cost**: for `small_b` each `last.pt` is ~1 GB and every save is a new LFS revision on the model repo. If storage is tight, raise `save_every_steps`, or clear `HUB_CKPT_REPO` and use *Save Version*. The `pretrained/` export (section 17, model-only, ~0.4 GB) is the artifact the ASR repo actually consumes.
- Other artifacts: `Khmer-LLaDA-Small/experiments/<RUN_NAME>/{metrics.jsonl, manifest.json}`, `training_curves.png`, `samples.json`.